# Game 2 - 1v1 Optimized Pair Trading (Coffee, G17)

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr
from statsmodels.tsa.stattools import adfuller

pd.set_option('display.float_format', '{:.4f}'.format)
print('imports OK')

## 2. Chargement des donnees

In [ ]:
CSV_PATH = 'coffee_dataset_G17.csv'

df_raw = pd.read_csv(CSV_PATH, index_col=0, parse_dates=True)
df_raw.index.name = 'Date'
df_raw = df_raw.sort_index()

df_clean = df_raw.dropna(thresh=int(0.80 * len(df_raw)), axis=1)
df_clean = df_clean.ffill().dropna()

ANCHOR = 'KC=F'
STOCKS = [c for c in df_clean.columns if c not in [ANCHOR, 'DBA']]

print(f'Dimensions : {df_clean.shape}')
print(f'Periode : {df_clean.index[0].date()} -> {df_clean.index[-1].date()}')
print(f'Anchor : {ANCHOR}')
print(f'Stocks : {STOCKS}')

## 3. Selection de la paire (Engle-Granger)

In [ ]:
log_prices = np.log(df_clean)
anchor_log = log_prices[ANCHOR]

FORMATION_END = '2022-12-31'
TRADING_START = '2023-01-01'

log_form = log_prices.loc[:FORMATION_END]

results = []
hedge_cache = {}

for stock in STOCKS:
    y = log_form[stock]
    x = log_form[ANCHOR]
    X = np.column_stack([np.ones(len(x)), x.values])
    beta = np.linalg.lstsq(X, y.values, rcond=None)[0]
    resid = y.values - X @ beta
    pval = adfuller(resid, regression='c')[1]
    hedge_cache[stock] = float(beta[1])
    rets_a = x.diff().dropna()
    rets_s = y.diff().dropna()
    common = rets_a.index.intersection(rets_s.index)
    corr, _ = pearsonr(rets_a.loc[common], rets_s.loc[common])
    results.append({'Asset': stock, 'Correlation': round(corr, 4),
                    'EG p-value': round(pval, 4), 'Hedge Ratio': round(beta[1], 4)})

coint_df = pd.DataFrame(results).sort_values('EG p-value')
coint_df

In [ ]:
PAIR = coint_df.iloc[0]['Asset']
HEDGE = hedge_cache[PAIR]
print(f'Paire selectionnee : KC=F / {PAIR}')
print(f'EG p-value : {coint_df.iloc[0]["EG p-value"]:.4f}')
print(f'Hedge ratio (beta) : {HEDGE:.4f}')

## 4. Optimisation du lookback (grid search)

In [ ]:
def compute_spread(log_p, anchor, pair, hedge):
    return log_p[pair] - hedge * log_p[anchor]


def rolling_zscore(series, window):
    mu = series.rolling(window).mean()
    sig = series.rolling(window).std()
    return (series - mu) / sig


def simple_backtest(spread_arr, z_arr, tc, z_entry=2.0, z_exit=0.0, capital=100_000):
    position = 0
    equity = capital
    eq_list = []
    entry_spread = None

    for i in range(len(z_arr)):
        z = z_arr[i]
        cur_spread = spread_arr[i]

        if position != 0:
            if (position == 1 and z >= z_exit) or (position == -1 and z <= z_exit):
                ret = (cur_spread - entry_spread) * position
                equity += ret / abs(entry_spread) * equity - tc * 2 * equity
                position = 0

        if position == 0:
            if z > z_entry:
                position = -1
                entry_spread = cur_spread
                equity -= tc * 2 * equity
            elif z < -z_entry:
                position = 1
                entry_spread = cur_spread
                equity -= tc * 2 * equity

        eq_list.append(equity)

    eq = np.array(eq_list)
    daily_ret = np.diff(eq) / eq[:-1]
    if daily_ret.std() == 0:
        return float('-inf')
    return (daily_ret.mean() - 0.04 / 252) / daily_ret.std() * np.sqrt(252)


log_form = log_prices.loc[:FORMATION_END]
spread_form = compute_spread(log_form, ANCHOR, PAIR, HEDGE)

lookbacks = list(range(20, 121, 5))
tc_grid = TX_COST_BPS / 10_000
sharpes = []

for lb in lookbacks:
    zs = rolling_zscore(spread_form, lb).dropna()
    sp = spread_form.loc[zs.index]
    s = simple_backtest(sp.values, zs.values, tc_grid)
    sharpes.append(s)

best_lb = lookbacks[np.argmax(sharpes)]
print(f'Lookback optimal : {best_lb} jours (IS Sharpe = {max(sharpes):.3f})')

## 5. Backtest out-of-sample

In [ ]:
Z_ENTRY     = 2.0
Z_EXIT      = 0.0
TX_COST_BPS = 10
CAPITAL     = 100_000
VOL_MULT    = 1.5
STOP_BASE   = 0.025

log_trade = log_prices.loc[TRADING_START:]
spread_trade = compute_spread(log_trade, ANCHOR, PAIR, HEDGE)
zscore_trade = rolling_zscore(spread_trade, best_lb).dropna()

spread_vol = spread_trade.rolling(best_lb).std()
avg_vol = spread_vol.rolling(252).mean()

common_idx = zscore_trade.index
spread_v = spread_vol.loc[common_idx]
avg_v = avg_vol.loc[common_idx]

tc = TX_COST_BPS / 10_000
position = 0
equity = CAPITAL
entry_spread = None
entry_equity = None
eq_list = []
pos_list = []
trade_log = []
entry_date = None

sp = spread_trade.loc[common_idx]
z_arr = zscore_trade.values
dates = common_idx

for i in range(len(dates)):
    z = z_arr[i]
    cur_spread = sp.iloc[i]
    vol_now = spread_v.iloc[i]
    avg_now = avg_v.iloc[i]

    vol_ratio = vol_now / avg_now if (avg_now and avg_now > 0) else 1.0
    stop_level = STOP_BASE * vol_ratio

    if position != 0:
        spread_move = (cur_spread - entry_spread) * position
        spread_ret = spread_move / abs(entry_spread) if entry_spread else 0

        stop_hit = spread_ret < -stop_level
        mean_rev = (position == 1 and z >= Z_EXIT) or (position == -1 and z <= Z_EXIT)

        if mean_rev or stop_hit:
            equity += spread_ret * equity - tc * 2 * equity
            trade_log.append({
                'open_date': entry_date,
                'close_date': dates[i],
                'direction': 'LONG' if position == 1 else 'SHORT',
                'pnl': round((equity - entry_equity), 2)
            })
            position = 0
            entry_spread = None

    if position == 0:
        vol_filter = (pd.isna(vol_now) or pd.isna(avg_now) or vol_ratio <= VOL_MULT)
        if vol_filter:
            if z > Z_ENTRY:
                position = -1
                entry_spread = cur_spread
                entry_equity = equity
                entry_date = dates[i]
                equity -= tc * 2 * equity
            elif z < -Z_ENTRY:
                position = 1
                entry_spread = cur_spread
                entry_equity = equity
                entry_date = dates[i]
                equity -= tc * 2 * equity

    eq_list.append(equity)
    pos_list.append(position)

portfolio = pd.DataFrame({
    'spread': sp.values,
    'zscore': z_arr,
    'position': pos_list,
    'equity': eq_list
}, index=dates)

trades = pd.DataFrame(trade_log)
print(f'Nombre de trades : {len(trades)}')
if len(trades) > 0:
    trades

## 6. Metriques

In [ ]:
equity_s = portfolio['equity']
daily_ret = equity_s.pct_change().dropna()
n_years = len(equity_s) / 252

total_ret = (equity_s.iloc[-1] / equity_s.iloc[0]) - 1
ann_ret   = (1 + total_ret) ** (1 / n_years) - 1
ann_vol   = daily_ret.std() * np.sqrt(252)
sharpe    = (daily_ret.mean() - 0.04 / 252) / daily_ret.std() * np.sqrt(252)

rolling_max = equity_s.cummax()
drawdown    = (equity_s - rolling_max) / rolling_max
max_dd      = drawdown.min()
calmar      = ann_ret / abs(max_dd) if max_dd != 0 else float('nan')

metrics = pd.DataFrame({
    'Total Return (%)' : [round(total_ret * 100, 2)],
    'Ann. Return (%)'  : [round(ann_ret * 100, 2)],
    'Ann. Vol (%)'     : [round(ann_vol * 100, 2)],
    'Sharpe Ratio'     : [round(sharpe, 3)],
    'Max Drawdown (%)' : [round(max_dd * 100, 2)],
    'Calmar Ratio'     : [round(calmar, 3)],
    '# Trades'         : [len(trades)],
}, index=[f'Pair Trade KC=F/{PAIR}'])

metrics

## 7. Performance

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

ax = axes[0]
ax.plot(portfolio.index, portfolio['equity'], label=f'Pair Trade KC=F/{PAIR}', linewidth=1.8)
ax.axhline(CAPITAL, color='grey', linestyle=':', linewidth=1)
ax.set_title('Valeur du portfolio (2023-2026)')
ax.set_ylabel('Valeur ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())

ax = axes[1]
ax.plot(portfolio.index, portfolio['zscore'], color='steelblue', linewidth=1, label='Z-score')
ax.axhline( Z_ENTRY, color='red',   linestyle='--', linewidth=0.9, label=f'+{Z_ENTRY}sig')
ax.axhline(-Z_ENTRY, color='green', linestyle='--', linewidth=0.9, label=f'-{Z_ENTRY}sig')
ax.axhline(0, color='black', linewidth=0.7)
ax.set_title(f'Z-score du spread (lookback={best_lb}j)')
ax.set_ylabel('Z-score')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig('figures/fig_performance_game2.png', dpi=150, bbox_inches='tight')
plt.show()